# Reporte comparativo: VLMs fine-tuneados sobre wildfire-prevention

Lee todos los `*.json` de `../results/` y genera tablas + plots comparativos.

**Pre-requisito**: haber entrenado y guardado los JSON de cada modelo (notebooks 01-04).

## 1. Cargar todos los resultados

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR  = NOTEBOOK_DIR.parent
RESULTS_DIR  = PROJECT_DIR / "results"

results = []
for f in sorted(RESULTS_DIR.glob("*.json")):
    with open(f, "r", encoding="utf-8") as fh:
        data = json.load(fh)
        data["_filename"] = f.name
        results.append(data)

print(f"Cargados {len(results)} modelos:")
for r in results:
    print(f"  - {r['model_short']:20s} ({r['model_name']})")

## 2. Tabla resumen

In [ ]:
rows = []
for r in results:
    t = r["training"]
    e = r["eval"]
    rows.append({
        "model":          r["model_short"],
        "params_total_M": (t.get("num_params_total") or 0) / 1e6,
        "params_train_M": (t.get("num_params_trainable") or 0) / 1e6,
        "vram_peak_GB":   t.get("vram_peak_gb"),
        "train_min":      t.get("training_time_min"),
        "loss_train":     t.get("loss_train_final"),
        "valid_json":     e.get("valid_json"),
        "overall":        e.get("overall"),
        "inf_s/sample":   e.get("inference_time_per_sample_s"),
    })

df = pd.DataFrame(rows).sort_values("params_total_M")
df.style.format({
    "params_total_M": "{:.0f}",
    "params_train_M": "{:.1f}",
    "vram_peak_GB":   "{:.2f}",
    "train_min":      "{:.1f}",
    "loss_train":     "{:.4f}",
    "valid_json":     "{:.3f}",
    "overall":        "{:.3f}",
    "inf_s/sample":   "{:.2f}",
}, na_rep="—")

## 3. Plot — overall accuracy por modelo

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
models   = df["model"].tolist()
overalls = df["overall"].fillna(0).tolist()

bars = ax.bar(models, overalls, color="steelblue", edgecolor="black")
ax.set_ylabel("Overall accuracy")
ax.set_title("Overall accuracy por modelo")
ax.set_ylim(0, 1)
ax.grid(axis="y", linestyle="--", alpha=0.5)

for bar, val in zip(bars, overalls):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.01,
            f"{val:.2f}", ha="center", va="bottom", fontsize=10)

plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

## 4. Plot — accuracy por campo (heatmap)

In [ ]:
FIELDS = ["risk_level", "dry_vegetation_present", "urban_interface",
          "steep_terrain", "water_body_present", "image_quality_limited"]

matrix = []
labels_y = []
for r in results:
    apf = r["eval"].get("accuracy_per_field") or {}
    row = [apf.get(f) if apf.get(f) is not None else np.nan for f in FIELDS]
    matrix.append(row)
    labels_y.append(r["model_short"])

matrix = np.array(matrix)

fig, ax = plt.subplots(figsize=(10, 1 + 0.6 * len(results)))
im = ax.imshow(matrix, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")

ax.set_xticks(range(len(FIELDS)))
ax.set_xticklabels(FIELDS, rotation=30, ha="right")
ax.set_yticks(range(len(labels_y)))
ax.set_yticklabels(labels_y)
ax.set_title("Accuracy por campo y modelo")

for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        v = matrix[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                    color="white" if v < 0.5 else "black", fontsize=9)
        else:
            ax.text(j, i, "—", ha="center", va="center", color="gray", fontsize=9)

plt.colorbar(im, ax=ax, label="accuracy")
plt.tight_layout()
plt.show()

## 5. Plot — Pareto: VRAM vs accuracy

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for _, row in df.iterrows():
    if pd.notna(row["vram_peak_GB"]) and pd.notna(row["overall"]):
        ax.scatter(row["vram_peak_GB"], row["overall"], s=200, alpha=0.7)
        ax.annotate(row["model"],
                    (row["vram_peak_GB"], row["overall"]),
                    textcoords="offset points", xytext=(8, 4), fontsize=10)

ax.set_xlabel("VRAM pico (GB)")
ax.set_ylabel("Overall accuracy")
ax.set_title("Frontera Pareto: VRAM vs accuracy")
ax.grid(linestyle="--", alpha=0.5)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 6. Plot — Tiempo de training vs accuracy

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for _, row in df.iterrows():
    if pd.notna(row["train_min"]) and pd.notna(row["overall"]):
        ax.scatter(row["train_min"], row["overall"], s=200, alpha=0.7)
        ax.annotate(row["model"],
                    (row["train_min"], row["overall"]),
                    textcoords="offset points", xytext=(8, 4), fontsize=10)

ax.set_xlabel("Tiempo de training (min)")
ax.set_ylabel("Overall accuracy")
ax.set_title("Coste de training vs accuracy")
ax.grid(linestyle="--", alpha=0.5)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 7. Plot — Latencia de inferencia por modelo

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

valid = df.dropna(subset=["inf_s/sample"]).sort_values("params_total_M")
ax.bar(valid["model"], valid["inf_s/sample"], color="orange", edgecolor="black")
ax.set_ylabel("Tiempo medio por sample (s)")
ax.set_title("Latencia de inferencia por modelo")
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

## 8. Tabla markdown exportable

Para pegar en un README, en un commit, en Notion...

In [ ]:
lines = ["| Modelo | Params (M) | VRAM (GB) | Train (min) | valid_json | overall | inf (s/sample) |",
         "|---|---:|---:|---:|---:|---:|---:|"]
for _, row in df.iterrows():
    def fmt(v, dec=2):
        return "—" if pd.isna(v) else f"{v:.{dec}f}"
    lines.append(f"| {row['model']} | {row['params_total_M']:.0f} | "
                 f"{fmt(row['vram_peak_GB'])} | {fmt(row['train_min'], 1)} | "
                 f"{fmt(row['valid_json'], 3)} | {fmt(row['overall'], 3)} | "
                 f"{fmt(row['inf_s/sample'])} |")

md = "\n".join(lines)
print(md)

# Guarda también a archivo
(RESULTS_DIR / "comparison_table.md").write_text(md, encoding="utf-8")
print(f"\n(También guardado en {RESULTS_DIR / 'comparison_table.md'})")